# Hotel Booking Cancellation Prediction

## 1. Project objective

The objective is to investigate historical hotel booking data and build a
machine learning model that estimates cancellation risk.

## 2. Business question

Can information available when a booking is created help us estimate
whether that booking will eventually be cancelled?

## 3. Dataset

- Source: TidyTuesday, 11 February 2020
- Original research: Antonio, Almeida, and Nunes (2019)
- Target: `is_canceled`
- Prediction point: booking creation

## 4. Data Preparation

- Download the data, read it with pandas
- Look at the data
- Make column names and values look uniform
- Check if all the columns ready correctly
- Check if the `is_canceled` needs any preparation

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [16]:
data = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv'

In [17]:
!wget $data -O hotel-bookings.csv

--2026-10-28 02:47:15--  https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16855599 (16M) [text/plain]
Saving to: ‘hotel-bookings.csv’

hotel-bookings.csv  100%[===================>]  16.07M   102MB/s    in 0.2s    

2026-10-28 02:47:15 (102 MB/s) - ‘hotel-bookings.csv’ saved [16855599/16855599]



In [18]:
df = pd.read_csv('hotel-bookings.csv')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [20]:
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [21]:
df.head().T

,0,1,2,3,4
hotel,resort_hotel,resort_hotel,resort_hotel,resort_hotel,resort_hotel
is_canceled,0,0,0,0,0
lead_time,342,737,7,13,14
arrival_date_year,2015,2015,2015,2015,2015
arrival_date_month,july,july,july,july,july
arrival_date_week_number,27,27,27,27,27
arrival_date_day_of_month,1,1,1,1,1
stays_in_weekend_nights,0,0,0,0,0
stays_in_week_nights,0,0,1,1,2
adults,2,2,1,1,2


## 5. EDA

- Leakage checks
- Duplicates check
- Inspect missing values
- Check numerical and categorical variables

Data leakage means the model gets to see information it wouldn’t have in real life. Our model should predict, at the moment someone makes a booking, “will this be cancelled?” Anything recorded later, or anything that is just the answer in disguise, is cheating.

### 5.1 Leakage and post-booking columns check

1) Timing test: what do the `reservation_status` and `reservation_status_date` say about the booking outcome `is_canceled`?

**Result**:  
Every “canceled” row has `is_canceled = 1`. Every “check-out” row has `is_canceled = 0`. A “no-show” counts as cancelled too. So `reservation_status` is the target written in different words, and drop it.

In [22]:
pd.crosstab(df['reservation_status'], df['is_canceled'], margins=True)

is_canceled,0,1,All
reservation_status,,,
canceled,0,43017,43017
check-out,75166,0,75166
no-show,0,1207,1207
All,75166,44224,119390


`reservation_status_date`: for cancelled bookings it is the cancellation date

In [23]:
df.groupby('reservation_status')['reservation_status_date'].agg(['min', 'max'])

,min,max
reservation_status,,
canceled,2014-10-17,2017-08-26
check-out,2015-07-01,2017-09-14
no-show,2015-07-02,2017-08-31


2) Empirical check: how well does ONE column alone predict the target?
   - AUC is a score for how well a column can separate cancelled from not cancelled
   - A tiny tree on each column; a suspiciously high AUC flags leakage.

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

**Result**:  
- `reservation_status` scored 1.000, a perfect score from one column. That is a red flag, and it matches the crosstab.
- Everything else scored 0.57 to 0.70, which is normal. Columns like `country`, `lead_time` and `deposit_type` help a bit but don’t give away the answer.
- `reservation_status_date` scored only 0.660 which looks innocent. But for cancelled bookings, that date is when the cancellation happened and the hotel only knows it afterward. It fails the timing check, so drop it.

In [37]:
def single_feature_auc(df, target="is_canceled", sample=30000, seed=42):
    d = df.sample(min(sample, len(df)), random_state=seed)
    y = d[target]
    rows = []
    for col in d.columns.drop(target):
        x = d[[col]].copy()

        if x[col].dtype == "object":
            x[col] = x[col].astype("category").cat.codes   # convert text-based categorical columns into numerical integers (label encoding)
        x = x.fillna(-1) # fill missing values with -1
        auc = cross_val_score(DecisionTreeClassifier(max_depth=5, random_state=seed), x, y, cv=3, scoring="roc_auc").mean()
        rows.append((col, round(auc, 3)))

    return pd.DataFrame(rows, columns=["column", "single_feature_auc"]).sort_values(
        "single_feature_auc", ascending=False)

single_feature_auc(df).head(12)

,column,single_feature_auc
29,reservation_status,1.000
12,country,0.696
1,lead_time,0.691
21,deposit_type,0.665
30,reservation_status_date,0.660
13,market_segment,0.641
28,total_of_special_requests,0.634
22,agent,0.624
26,adr,0.605
19,assigned_room_type,0.605


3) Post-booking suspects: `booking_changes` happens after booking. If the model uses this column, it is peeking into the future, so drop it.

In [26]:
df.groupby('is_canceled')['booking_changes'].describe()

,count,mean,std,min,25%,50%,75%,max
is_canceled,,,,,,,,
0,75166.0,0.293364,0.736266,0.0,0.0,0.0,0.0,21.0
1,44224.0,0.098340,0.451008,0.0,0.0,0.0,0.0,16.0


**Result**:  
18.8% (roughly 95 out of 100 bookings) with different assigned room were not cancelled. The hotel assigns rooms close to arrival, so cancelled guest mostly never gets one. Different room means the guest probably showed up. The hotel knows `assigned_room_type` after the booking is made, so drop it.

In [33]:
pd.crosstab(df['reserved_room_type'] != df['assigned_room_type'], df['is_canceled'], normalize="columns").round(3)

is_canceled,0,1
row_0,,
False,0.812,0.982
True,0.188,0.018


4) Counterintuitive but legitimate: the hotel knows the `deposit_type` when booking is made, so it is not leakage.

In [36]:
pd.crosstab(df['deposit_type'], df['is_canceled'], normalize="index").round(3)

is_canceled,0,1
deposit_type,,
no_deposit,0.716,0.284
non_refund,0.006,0.994
refundable,0.778,0.222


## 6. Cleaning Decision

- Exclude leakage columns
- Deduplicate
- Handle missing values

### 6.1 Drop the leaky and post-booking columns:

- `reservation_status`
- `reservation_status_date`
- `booking_changes`
- `assigned_room_type`

In [46]:
LEAKY = ['reservation_status', 'reservation_status_date']
POST_BOOKING = ['booking_changes', 'assigned_room_type']

df_clean = df.drop(columns=LEAKY + POST_BOOKING)
feature_cols = [c for c in df_clean.columns if c != 'is_canceled']
feature_cols

['hotel',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'deposit_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests']

### 5.2 Duplicates check

In [45]:
# Count extra occurences of exact duplicate rows
duplicate_rows = int(df_clean.duplicated().sum())
duplicate_rows

34050